<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 2 - Claudia

## Validate source filename sequence

In [1]:
from pathlib import Path
import re

source_dir = Path("corpus/00_sources")

filename_pattern = re.compile(r"^(\d{3}|1000)\.txt$")

files = sorted(source_dir.glob("*.txt"), key=lambda path: int(path.stem))
filenames = [path.name for path in files]

invalid_filenames = [
    name for name in filenames
    if not filename_pattern.fullmatch(name)
]

parsed_numbers = [
    int(path.stem)
    for path in files
    if filename_pattern.fullmatch(path.name)
]

expected_numbers = list(range(1, 1001))
expected_filenames = [
    f"{number:03d}.txt" if number < 1000 else "1000.txt"
    for number in expected_numbers
]

missing_filenames = sorted(set(expected_filenames) - set(filenames))
extra_filenames = sorted(set(filenames) - set(expected_filenames))

is_uninterrupted = (
        parsed_numbers == expected_numbers
        and not invalid_filenames
        and not missing_filenames
        and not extra_filenames
)

print(f"Files found: {len(filenames)}")
print(f"First file: {filenames[0] if filenames else 'None'}")
print(f"Last file: {filenames[-1] if filenames else 'None'}")
print(f"Invalid filenames: {invalid_filenames or 'None'}")
print(f"Missing filenames: {missing_filenames or 'None'}")
print(f"Extra filenames: {extra_filenames or 'None'}")
print(f"Uninterrupted 001.txt–1000.txt sequence: {is_uninterrupted}")

assert is_uninterrupted, "Source filenames are not an uninterrupted sequence from 001.txt to 1000.txt."

Files found: 1000
First file: 001.txt
Last file: 1000.txt
Invalid filenames: None
Missing filenames: None
Extra filenames: None
Uninterrupted 001.txt–1000.txt sequence: True


## Recover original transcript IDs for numbered source files

In [2]:
      from collections import defaultdict
from hashlib import sha256
from pathlib import Path

import pandas as pd

source_dir = Path("corpus/00_sources")
original_transcripts_dir = Path("../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini")
mapping_output_path = Path("corpus/00_sources_mapping.tsv")


def file_sha256(path: Path) -> str:
    """Return the SHA-256 hash of a file's raw bytes."""
    digest = sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


source_files = sorted(source_dir.glob("*.txt"), key=lambda path: int(path.stem))
original_files = sorted(original_transcripts_dir.glob("*.txt"))

original_files_by_hash = defaultdict(list)
for original_file in original_files:
    original_files_by_hash[file_sha256(original_file)].append(original_file)

mapping_rows = []
for source_file in source_files:
    source_hash = file_sha256(source_file)
    matching_original_files = original_files_by_hash.get(source_hash, [])
    matching_original_names = [path.name for path in matching_original_files]
    matching_corpus_ids = [path.stem for path in matching_original_files]

    if len(matching_original_files) == 1:
        status = "matched"
    elif len(matching_original_files) == 0:
        status = "unmatched"
    else:
        status = "ambiguous"

    mapping_rows.append(
        {
            "source_number": int(source_file.stem),
            "source_file": source_file.as_posix(),
            "matched_original_file": matching_original_files[0].as_posix() if len(matching_original_files) == 1 else None,
            "corpus_id": matching_corpus_ids[0] if len(matching_corpus_ids) == 1 else None,
            "match_count": len(matching_original_files),
            "status": status,
            "all_matching_original_files": "; ".join(matching_original_names),
            "sha256": source_hash,
        }
    )

source_mapping_df = pd.DataFrame(mapping_rows)
source_mapping_df.to_csv(mapping_output_path, sep="	", index=False)

status_counts = source_mapping_df["status"].value_counts().to_dict()
print(f"Phase 2 source files: {len(source_files)}")
print(f"Phase 1 candidate files: {len(original_files)}")
print(f"Matched uniquely: {status_counts.get('matched', 0)}")
print(f"Unmatched: {status_counts.get('unmatched', 0)}")
print(f"Ambiguous: {status_counts.get('ambiguous', 0)}")
print(f"Mapping written to: {mapping_output_path}")

source_mapping_df

Phase 2 source files: 1000
Phase 1 candidate files: 1000
Matched uniquely: 999
Unmatched: 1
Ambiguous: 0
Mapping written to: corpus/00_sources_mapping.tsv


,source_number,source_file,matched_original_file,corpus_id,match_count,status,all_matching_original_files,sha256
0,1,corpus/00_sources/001.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,eH3giaIzONA,1,matched,eH3giaIzONA.txt,408fb3d07b5a90f22dc0c27c5009740d9d12fda275f303...
1,2,corpus/00_sources/002.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,xFrGuyw1V8s,1,matched,xFrGuyw1V8s.txt,b55da182ebea1985d0bdd7022760412363449d11588d7a...
2,3,corpus/00_sources/003.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,y3KJ7d2qBoA,1,matched,y3KJ7d2qBoA.txt,0a40c55c88477b2d6492f689143a61bab3ed487d1946cd...
3,4,corpus/00_sources/004.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,4fndeDfaWCg,1,matched,4fndeDfaWCg.txt,d9e8ef74e67b34e2d4498664dc8216a148667954087daa...
4,5,corpus/00_sources/005.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,R7UrFYvl5TE,1,matched,R7UrFYvl5TE.txt,e38e1822a047a94c8b6240d83e2d917c54f2cde3a685d6...
...,...,...,...,...,...,...,...,...
995,996,corpus/00_sources/996.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,yKNxeF4KMsY,1,matched,yKNxeF4KMsY.txt,aa94b4d14fb5a8d8e461e286ce8dc200d257a2f7b7f8cb...
996,997,corpus/00_sources/997.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,V1Pl8CzNzCw,1,matched,V1Pl8CzNzCw.txt,692e8ed96b40aae490445f3523992166a704b51d4620d6...
997,998,corpus/00_sources/998.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,mRD0-GxqHVo,1,matched,mRD0-GxqHVo.txt,2044c3a247b15530a3d09151bc86db94a59f1515e75fe9...
998,999,corpus/00_sources/999.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,nyuo9-OjNNg,1,matched,nyuo9-OjNNg.txt,7f031bed3d7d6e70b93a9cad25b3b4d8a35c989842a803...
